# Querying a remote HSDS index

The same tour as [queries.ipynb](queries.ipynb), but against an index that
already exists on an HSDS server instead of a local `.h5` built from `sample/`.

Two things change and nothing else does:

- **There is no ingest step.** The domain was built by whoever runs the
  archive; this notebook is read-only.
- **Every column read is a network round trip.** The `scanned_rows` and
  `pv_reads` counters stop being trivia and start being the thing you tune.

The query code itself is identical — `Store` never learns which backend it got,
because both are `h5py`-shaped.

## Connect

Configuration comes from `ARCHIVER_*` environment variables or a `.env` file;
the cell below lets them win and falls back to the local development server.
Set `backend="hsds"` explicitly rather than relying on `auto` — `auto` guesses
from the domain name, and a domain like `/home/admin/clara/archiver.h5` ends in
`.h5` even though it is not a file on this machine.

(`.hscfg` in the repo root configures the `hsls`/`hstouch` command-line tools.
It is *not* read here — this package is configured through `ARCHIVER_*`.)

In [ ]:
import os
from datetime import datetime, timedelta

from archiver_search.config import Settings
from archiver_search.models import (
    CameraFilter,
    Clause,
    Group,
    SearchRequest,
    SeriesRequest,
    Sort,
    TimeRange,
)
from archiver_search.store import QueryError, Store

settings = Settings(
    domain=os.environ.get("ARCHIVER_DOMAIN", "/home/admin/clara/archiver.h5"),
    hsds_endpoint=os.environ.get("ARCHIVER_HSDS_ENDPOINT", "http://localhost:5101"),
    hsds_username=os.environ.get("ARCHIVER_HSDS_USERNAME", "admin"),
    hsds_password=os.environ.get("ARCHIVER_HSDS_PASSWORD", "admin"),
    backend="hsds",
)

print(f"domain   {settings.domain}")
print(f"endpoint {settings.hsds_endpoint}")

Opening the store does no work; the first query pulls the index snapshot — the
timestamp and camera columns plus the PV catalogue — and caches it. That one
fetch is why the first call below is slower than every call after it.

If this raises, the usual causes are a wrong endpoint, credentials the server
does not recognise, or a domain that does not exist. `hsls -e <endpoint>
/home/admin/clara/` lists what is actually there.

In [ ]:
store = Store(settings=settings)
meta = store.meta()

print(f"{meta['n_rows']} rows, {meta['pv_count']} PVs")
print(f"schema v{meta['schema_version']}, built {meta['created_utc']}")
print(f"{meta['time_from']}  ..  {meta['time_to']}")
print()
for cam in meta["cameras"]:
    print(f"{cam['name']:<22} {cam['area']:<4} {cam['n_rows']:>6} rows   {cam['first']} .. {cam['last']}")

## Adapt to whatever this archive holds

Unlike the sample notebook, we cannot assume which PVs exist or what dates are
covered. Derive both from the index: pick PVs through the catalogue, and build
time windows from the reported bounds.

In [ ]:
def pick(preferred, **catalogue_filters):
    """The preferred PV if this archive has it, else the first catalogue match."""
    try:
        return store.resolve_pv(preferred).name
    except QueryError:
        candidates = store.list_pvs(**catalogue_filters)
        if not candidates:
            raise
        print(f"note: {preferred} is not in this archive, using {candidates[0].name}")
        return candidates[0].name


MAGNET = pick("CLA-S04-MAG-QUAD-01:SETI", family="MAG", field_="SETI")
PHASE = pick("CLA-L01-RF-CTRL-01:getPhase", field_="getPhase")


def iso(text):
    return datetime.fromisoformat(text.replace("Z", "+00:00"))


LAST = iso(meta["time_to"])
RECENT = LAST - timedelta(days=1)  # the final day covered by the index

print(f"magnet  {MAGNET}")
print(f"phase   {PHASE}")
print(f"window  {RECENT.isoformat()}  ..  {LAST.isoformat()}")

The catalogue also carries the min and max recorded for each PV at ingest, which
is how you pick a threshold that will match something without a round trip to
find out.

In [ ]:
info = store.resolve_pv(MAGNET)
print(f"{info.name}  {info.min} .. {info.max}")

# A range covering the lower half of what was ever recorded.
LO, HI = info.min, (info.min + info.max) / 2
print(f"querying for {LO} .. {HI}")

## Query 1 — one PV in a range

One predicate on one PV is exactly one hyperslab read, however many rows match.
Over HSDS that is one HTTP request for a contiguous slice of one column.

In [ ]:
def show(result, *, n=5):
    """Print the headline counters and the first few rows of a search result."""
    print(
        f"{result['total']} row(s) · scanned {result['scanned_rows']} · "
        f"{result['pv_reads']} PV read(s) · {result['elapsed_ms']} ms"
    )
    for row in result["rows"][:n]:
        values = "  ".join(
            f"{name.split(':')[-1]}={value:.3f}" if value is not None else f"{name}=nan"
            for name, value in row["values"].items()
        )
        print(f"  {row['id']:>6}  {row['timestamp']}  {row['camera']:<22} {values}")


q1 = store.search(
    SearchRequest(
        filters=Group(clauses=[Clause(pv=MAGNET, op="between", value=[LO, HI])]),
        select=[MAGNET],
        limit=5,
    )
)
show(q1)

## The cache is the whole point

Run the identical query again. `pv_reads` drops to 0: the column slice is
already in the process, so nothing crosses the network at all. This is why a
`Store` is shared for the life of the API process rather than opened per
request.

How much wall-clock that saves depends entirely on what is between you and the
server. Against a local HSDS the saving is small — the round trip was cheap to
begin with. Against a real remote endpoint the read that did *not* happen is the
dominant cost of the query.

`ARCHIVER_PV_CACHE_MIB` (default 256) bounds that cache;
`ARCHIVER_CACHE_TTL_SECONDS` (default 30) is how long before the index snapshot
is re-checked against the domain, which is what lets a long-lived reader notice
rows appended by a later ingest.

In [ ]:
again = store.search(
    SearchRequest(
        filters=Group(clauses=[Clause(pv=MAGNET, op="between", value=[LO, HI])]),
        select=[MAGNET],
        limit=5,
    )
)
show(again, n=0)
print(f"\ncold {q1['elapsed_ms']} ms  ->  warm {again['elapsed_ms']} ms")

## Query 2 — narrow by time first

The time window is the cheapest thing you can do and the most valuable one over
a network: it is a binary search over the cached timestamp column, and it
shrinks every hyperslab that follows. Compare `scanned_rows` here against the
unwindowed query above.

Camera and area filters are likewise free — a mask over a cached integer column.

In [ ]:
windowed = store.search(
    SearchRequest(
        time=TimeRange(from_=RECENT, to=LAST),
        filters=Group(clauses=[Clause(pv=MAGNET, op="between", value=[LO, HI])]),
        select=[MAGNET],
        limit=3,
    )
)
show(windowed)

# Time and camera alone never touch a PV: pv_reads stays at 0.
area = meta["cameras"][0]["area"]
by_area = store.search(SearchRequest(cameras=CameraFilter(areas=[area]), limit=0))
print(f"\narea {area}:")
show(by_area)

## Query 3 — several conditions at once

Clauses in a `Group` combine with `and` or `or`, and groups nest. The PVs a
query mentions are prefetched in parallel — `ARCHIVER_READ_WORKERS` (default 8)
sets how many hyperslab reads are in flight at once, so a query naming eight
PVs costs roughly one round trip rather than eight.

`eq` uses an absolute tolerance (`tol`, default `1e-9`) because these are floats
read back from a control system.

In [ ]:
nested = store.search(
    SearchRequest(
        time=TimeRange(from_=RECENT, to=LAST),
        filters=Group(
            op="and",
            clauses=[
                Clause(pv=MAGNET, op="between", value=[LO, HI]),
                Group(
                    op="or",
                    clauses=[
                        Clause(pv_pattern="*:getPhase", match="any", op="notnan"),
                        Clause(pv=PHASE, op="isnan"),
                    ],
                ),
            ],
        ),
        select=[MAGNET, PHASE],
        sort=Sort(by="timestamp", order="desc"),
        limit=5,
    )
)
show(nested)

## Query 4 — a time series

`series` returns one trace per PV over the matching rows. Ask for a glob and you
get every PV it matches, each one a hyperslab; keep the time window tight.

Traces longer than `max_points` are min/max decimated **server-side**, so a year
of data does not become a year of JSON.

In [ ]:
result = store.series(
    SeriesRequest(
        pv_pattern="*:getPhase",
        time=TimeRange(from_=RECENT, to=LAST),
        max_points=500,
    )
)

print(f"{result['total']} matching row(s) · {result['elapsed_ms']} ms")
for trace in result["series"]:
    values = [v for v in trace["v"] if v is not None]
    span = f"{min(values):>9.2f} .. {max(values):>8.2f}" if values else "(all nan)"
    print(f"  {trace['pv']:<30} {len(trace['t']):>5} pts  {span}  decimated={trace['decimated']}")

## One row in full

`row(id)` attaches every PV to a single capture. That is one read per PV, so it
is the most expensive call in this notebook — fine for inspecting a row you have
already found, wrong as a way to scan.

In [ ]:
row = store.row(q1["rows"][0]["id"])

print(f"{row['camera']}  {row['timestamp']}  ({row['n_images']} image(s))")
print(f"source   {row['source_file']}")
print(f"capture  {row['capture_path']}")
print(f"mounted  {row['has_image']}")
print(f"\n{len(row['values'])} PV values, e.g.")
for name in sorted(row["values"])[:5]:
    print(f"  {name:<30} {row['values'][name]}")

**Images are referenced, not stored.** The capture files live on the CLARA
storage, not in HSDS and not alongside this index. What a row carries is where
its capture sits *relative to the acquisition host's archive root*, so a reader
can mount that archive wherever it likes. The datasets inside are named
`Capture000001` … `Capture{n_images:06d}`:

```python
import h5py

with h5py.File(f"{image_root}/{row['capture_path']}") as f:
    image = f["Capture000001"][:]   # greyscale
```

`has_image` is `True` only when *this* machine has the file, which it decides by
joining the path onto `Settings(image_root=...)` / `ARCHIVER_IMAGE_ROOT`. An
index read over HSDS from a workstation will normally say `False`: you have the
metadata without the pixels, which is the point of indexing them separately.

## Bulk retrieval

`iter_rows` streams every matching row in chunks, ignoring the page limit and
bounded instead by `ARCHIVER_MAX_EXPORT_ROWS`. Row selection has already
happened by this point, so the cost here is hydrating strings, not re-querying.
Keep a window on it against a real archive.

In [ ]:
request = SearchRequest(
    time=TimeRange(from_=RECENT, to=LAST),
    select=[MAGNET],
    limit=0,
)

total = 0
for block in store.iter_rows(request, chunk=100):
    total += len(block)
print(f"{total} row(s) streamed")

In [ ]:
store.close()  # releases the HSDS connection and drops the caches

## Pointing this somewhere else

Nothing above is specific to the endpoint. Set the environment and re-run:

```bash
export ARCHIVER_DOMAIN=/home/admin/clara/archiver.h5
export ARCHIVER_HSDS_ENDPOINT=https://hsds.example.ac.uk
export ARCHIVER_HSDS_USERNAME=...
export ARCHIVER_HSDS_PASSWORD=...
export ARCHIVER_BACKEND=hsds
```

To run a local HSDS to try this against, see `scripts/bootstrap.sh` and
`scripts/hsds_start.sh`, then ingest the sample into it:

```bash
scripts/hsds_start.sh
.venv/bin/python -m archiver_search.ingest sample
```

If you would rather not import this package at all, the service exposes the same
queries over HTTP — see [queries_http.ipynb](queries_http.ipynb).